# Phase 2 on Colab — main experiment + false-negative audit

Sequencer for [`PHASE2_PLAN.md`](../PHASE2_PLAN.md). This notebook is deliberately thin: experiment logic belongs in the tested `embeded/` Python modules, not duplicated in notebook cells.

> **Current status (2026-09-27):** gate rule D1 is frozen and `embeded.train` / `embeded.evaluate` are implemented and unit-tested offline. **No Phase 2 training has been run yet** — the cells below still need a T4, regenerated Phase 1 artifacts and the scored blind audit before they produce results.

## Order of work
1. Start a Colab **T4 GPU** runtime, clone the code version containing the Phase 2 implementation, configure optional Hugging Face artifact sync, and verify the environment.
2. Run tests and inspect/retrieve the frozen Phase 1 artifacts. Resolve the G1 rule explicitly; do not bypass the current absolute-margin gate or erase its historical FAIL.
3. Run the blind C2/C3 50+50 audit, label the pairs without opening the key, and score it before training.
4. Run a tiny end-to-end training smoke test.
5. Run C0 once and C1/C2/C3 for seeds 13, 14, 15. Checkpoint outputs after each stage.
6. Review validation-selected thresholds and test results, create the Phase 2 table, and record the actual work in `PHASE2_PLAN.md` and `report/measurements.md`.

Never use the test split to tune a threshold or hyperparameter. Keep Hugging Face tokens in Colab Secrets, not notebook code.

## 0. Runtime and code checkout

In Colab choose **Runtime → Change runtime type → T4 GPU**. Set `REPO_REF` below to the branch/commit that contains the approved Phase 2 code. This notebook defaults to the Arena work branch; use `main` after the changes are merged.

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/ksu-gitreaper807/EmbedEd.git'
REPO_REF = 'arena/01a0e257-embeded'  # branch carrying the D1 gate rule + the Phase 2 runner; use main once merged
ROOT = '/content/EmbedEd'

def _git(*args, check=True):
    p = subprocess.run(['git', *args], cwd=ROOT, capture_output=True, text=True)
    if check and p.returncode:
        raise SystemExit(f"git {' '.join(args)} exited {p.returncode}\n{p.stdout}{p.stderr}")
    return p.stdout.strip()

if not os.path.isdir(os.path.join(ROOT, '.git')):
    subprocess.run(['git', 'clone', '--branch', REPO_REF, '--single-branch', REPO_URL, ROOT], check=True)
else:
    _git('fetch', 'origin', REPO_REF)
    ahead = _git('rev-list', '--count', 'FETCH_HEAD..HEAD', check=False)
    if ahead not in ('', '0'):
        raise SystemExit(
            f'{ahead} local commit(s) on this VM are not on origin/{REPO_REF}. Refusing to move the '
            'branch and orphan them: run notebook section 8 to push them (or `git rebase FETCH_HEAD` '
            'and push by hand), then re-run this cell.')
    _git('checkout', '-B', REPO_REF, 'FETCH_HEAD')
%cd /content/EmbedEd
print('Checked out:', _git('rev-parse', '--short', 'HEAD'))


## 1. Configure caches, install pinned dependencies, and verify GPU

Do not install/pin PyTorch separately: use the CUDA-matched build supplied by Colab. The cache pull is optional; without a configured HF dataset repo this session starts with local artifacts only.

In [ ]:
import os, sys
ROOT = '/content/EmbedEd'
ART = '/content/embeded-artifacts'
os.makedirs(ART, exist_ok=True)
os.environ['HF_HOME'] = '/content/hf-home'
os.environ['EMBEDED_ARTIFACTS'] = ART
os.environ['EMBEDED_REPORT'] = ROOT + '/report/measurements.md'
try:
    from google.colab import userdata
    for key in ('EMBEDED_HF_REPO_ID', 'EMBEDED_HF_REPO_TYPE', 'EMBEDED_HF_REVISION', 'EMBEDED_HF_SUBDIR', 'HF_TOKEN'):
        try: value = userdata.get(key)
        except Exception: value = None
        if value: os.environ.setdefault(key, value)
except Exception:
    pass
%pip install -q -U transformers==4.46.3 datasets==2.20.0 sentence-transformers==3.0.1 rank-bm25==0.2.2 umap-learn==0.5.6 scikit-learn==1.6.1 gradio==5.49.1 pytest==8.3.2
import torch, transformers, datasets, sentence_transformers, sklearn
print(f'torch={torch.__version__} cuda={torch.cuda.is_available()} | transformers={transformers.__version__} | datasets={datasets.__version__} | sklearn={sklearn.__version__}')
if not torch.cuda.is_available(): raise RuntimeError('GPU is required. Select a Colab T4 runtime before continuing.')
gpu = torch.cuda.get_device_name(0)
print('GPU:', gpu)
assert 'T4' in gpu.upper(), f'Expected the measured T4 configuration; got {gpu}. Re-measure throughput before using the frozen budget.'
from scripts.hf_artifacts import main as hf_main
hf_main(['pull', '--if-configured'])

## 2. Preflight: artifacts, tests, and gate G1 under decision D1

Order matters: the gate reads the Phase 1 artifacts, so they are checked first — a fresh VM has none and `hardness_diagnostics` would otherwise die inside `numpy` with a bare `FileNotFoundError`.

Gate G1 now uses the **owner-approved scale-aware rule (decision D1, recorded 2026-09-27)**: `C1 < C2 ≤ C3` **and** standardised gap `d(C1→C2) ≥ HARDNESS_D_MIN = 0.5`, with the corpus percentiles reported alongside. The legacy `HARDNESS_MARGIN = 0.02` reading is still printed and recorded, but it is no longer the verdict; the 2026-09-26 FAIL stays verbatim in `report/measurements.md`. Do not edit `HARDNESS_D_MIN` to get past a FAIL — a failed manipulation means stop and fix the mining (SCOPE P6-3).

In [ ]:
import subprocess, sys
from embeded import settings as S

# 1. artifacts first: the gate and the diagnostics both need them, and the
#    message below is the one the modules print too (embeded.hardcheck)
required = [S.artifact('fragments.jsonl'), S.artifact('corpus_emb.npy'),
            S.artifact('corpus_emb.meta.json')] + [S.artifact(f'triples_{c}.jsonl') for c in ('C1', 'C2', 'C3')]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        'Phase 1 artifacts missing — regenerate all of them before the gate (PHASE2_PLAN §2 item 2):\n  '
        + '\n  '.join(missing)
        + '\n  python -m embeded.data.prepare_data --hf --verify-spec'
          '\n  python -m embeded.mining.semantic_index   # GPU encode, ~4 min on a T4'
          '\n  python -m embeded.negatives               # all three conditions together, ~15-30 min')

# 2. offline tests, then the scale-of-the-space view, then the gate
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'embeded/tests'], check=True)
subprocess.run([sys.executable, '-m', 'scripts.hardness_diagnostics', '--examples', '3'], check=True)
subprocess.run([sys.executable, '-m', 'embeded.hardcheck'], check=True)   # exit 1 = FAIL => stop
hf_main(['push', '--if-configured', '--include-report'])
print('G1 PASS under D1 — this run was appended to the G1 history next to the 2026-09-26 FAIL.')


### Artifact check

Only continue if the exact frozen corpus/triples are present and verified for the current code/settings version. Regenerate/retrieve them with the Phase 0/1 notebook if missing or stale. If anything about mining changes, regenerate **all three conditions together**, then rerun tests, gate, and diagnostics.

In [ ]:
from pathlib import Path
from embeded import settings as S
required = [S.artifact('fragments.jsonl'), S.artifact('corpus_emb.npy')] + [S.artifact(f'triples_{c}.jsonl') for c in ('C1','C2','C3')]
missing = [str(p) for p in required if not p.exists()]
if missing: raise FileNotFoundError('Missing Phase 1 artifacts; restore/regenerate and validate them before training:\n' + '\n'.join(missing))
print('Artifacts present for VERSION', S.VERSION)
import json as _json
_summary = S.ARTIFACTS / 'mining_summary.json'
if _summary.exists():
    _v = (_json.loads(_summary.read_text()).get('_run') or {}).get('version')
    print('mining_summary.json _run.version =', _v)
    if _v != S.VERSION:
        raise SystemExit(
            f'artifacts were mined by {_v!r} but this checkout is {S.VERSION!r}. '
            'embeded.train/evaluate refuse to run on them (exit 1, no traceback).\n'
            '  back up triples_C*.jsonl + mining_summary.json, then re-mine all three together:\n'
            '    python -m embeded.negatives --strategies random,bm25,semantic --force\n'
            '  and diff the new triples against the backup: if they are identical the blind\n'
            '  audit key still matches and its labels survive; if not, re-sample and re-label.')
else:
    print('mining_summary.json absent - run `python -m embeded.negatives` before training')
print('NOTE: confirm metadata/hashes and the shared anchors/positives/k; file existence alone is not an identity check.')

## 3. Make and complete the blind false-negative audit (50 C2 + 50 C3)

Generate the shuffled, condition-blind sample. Label each pair `clone`, `not_clone`, or `unsure` by functional equivalence. Do not inspect `audit_key.csv` until scoring. Download only the pair sheet and blank labels; keep the key private. After labeling, upload the completed CSV in the next cell. This audit is a Phase 2 entry criterion, not a result to fill in from token-Jaccard proxies.

In [ ]:
from pathlib import Path
import subprocess, sys
from google.colab import files
audit_dir = Path(ART) / 'audit'
pairs_file, labels_file, key_file = (audit_dir / n for n in ('audit_pairs.md', 'audit_labels.csv', 'audit_key.csv'))
if not (pairs_file.exists() and labels_file.exists() and key_file.exists()):
    subprocess.run([sys.executable, '-m', 'scripts.audit_sample', 'make', '--n', '50', '--seed', '13'], check=True)
else:
    print('Existing audit artifacts found; reusing them to avoid overwriting labels.')
hf_main(['push', '--if-configured'])  # checkpoint the blind key + blank sheet before labeling
# (the s' package is no longer staged: push skips sprime/* and *.zip)
files.download(str(pairs_file))
files.download(str(labels_file))
print('Do not download/open audit_key.csv until after labels are final.')


### After labeling offline: upload the completed labels and score

Upload the filled `audit_labels.csv` only. Keep `audit_key.csv` unchanged and do not edit it by hand. Scoring writes the blinded C2/C3 rates, Wilson intervals, unsure upper bounds, JSON result, and report section. Review the audit result and document how it affects interpretation before allowing training.

In [ ]:
from google.colab import files
uploaded = files.upload()
if 'audit_labels.csv' not in uploaded: raise ValueError('Upload the completed file named audit_labels.csv')
from pathlib import Path
import subprocess, sys
(Path(ART) / 'audit').mkdir(parents=True, exist_ok=True)
(Path(ART) / 'audit' / 'audit_labels.csv').write_bytes(uploaded['audit_labels.csv'])
subprocess.run([sys.executable, '-m', 'scripts.audit_sample', 'score'], check=True)
hf_main(['push', '--if-configured', '--include-report'])


## 4. Smoke test the Phase 2 runner

`embeded.train` and `embeded.evaluate` are implemented and unit-tested offline. `--smoke` runs the whole harness on a tiny subset — load → forward/backward → checkpoint save/reload (verified tensor-by-tensor) → validation scoring → test metrics — and is a harness check, never an experiment result. Full runs use the frozen recipe in `settings.py` (`MAX_LEN = 512`, `BATCH = 8`, `TRAIN_PAIRS_CAP = 32,000`, `EPOCHS = 1`, fp16 autocast).

In [ ]:
import os, subprocess, sys
# Re-establish the environment: `-m embeded.*` resolves the package from the cwd and
# reads EMBEDED_ARTIFACTS from the environment, so a fresh runtime that skipped cell 1
# must not silently train against the in-repo default artifacts dir.
ROOT, ART = '/content/EmbedEd', '/content/embeded-artifacts'
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)   # os.chdir alone does not make `embeded` importable
os.environ.setdefault('EMBEDED_ARTIFACTS', ART)
os.environ.setdefault('EMBEDED_REPORT', ROOT + '/report/measurements.md')
os.environ['PYTHONPATH'] = ROOT + os.pathsep + os.environ.get('PYTHONPATH', '')
from importlib.util import find_spec
if find_spec('embeded.train') is None or find_spec('embeded.evaluate') is None:
    raise RuntimeError('Phase 2 runner modules are missing from this checkout - check REPO_REF in cell 0.')
# stderr is inherited, so the runner's own message (not just CalledProcessError) is on screen.
p = subprocess.run([sys.executable, '-m', 'embeded.train', '--condition', 'C1', '--seed', '13', '--smoke'])
if p.returncode:
    raise SystemExit(f'smoke run exited {p.returncode} - the reason is printed above. Do not start '
                     'the nine main runs until this passes.')
# trains a few steps, reloads the checkpoint, then evaluates on a tiny subset of
# both splits. Expect ~3 min on a T4; the numbers it prints mean nothing.

## 5. Evaluate the untuned C0 baseline once

C0 uses the untouched pretrained encoder and the same tokenizer, mean pooling, max length, and scoring path as the trained runs. Select any F1 threshold on validation only, apply it to test once, and retain predictions/configuration. Do not fine-tune C0.

In [ ]:
import os, subprocess, sys
# Re-establish the environment: `-m embeded.*` resolves the package from the cwd and
# reads EMBEDED_ARTIFACTS from the environment, so a fresh runtime that skipped cell 1
# must not silently train against the in-repo default artifacts dir.
ROOT, ART = '/content/EmbedEd', '/content/embeded-artifacts'
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)   # os.chdir alone does not make `embeded` importable
os.environ.setdefault('EMBEDED_ARTIFACTS', ART)
os.environ.setdefault('EMBEDED_REPORT', ROOT + '/report/measurements.md')
os.environ['PYTHONPATH'] = ROOT + os.pathsep + os.environ.get('PYTHONPATH', '')
p = subprocess.run([sys.executable, '-m', 'embeded.evaluate', '--condition', 'C0', '--seed', '13'])
if p.returncode:
    raise SystemExit(f'C0 evaluation exited {p.returncode} - the reason is printed above.')

## 6. Run the nine main training jobs (3 conditions x 3 seeds)

This sequential loop is intentionally fail-fast, and it now opens with a preflight: the runners report a missing artifact or a stale `mining_summary.json` as `SystemExit(message)` plus exit code 1 with **no traceback**, which a bare `CalledProcessError` hides. The preflight checks the artifact paths, the triple counts, the version fingerprint and the GPU before the first 40-minute run, and the wrapper names those three failures when a child still dies. It also re-establishes `cwd` and `EMBEDED_ARTIFACTS`, because `python -m embeded.train` resolves both from the environment - a fresh runtime that skipped cell 1 would otherwise train against the in-repo default artifacts directory.

disconnects should resume from verified checkpoints, not silently restart or mix versions.

In [ ]:
import os, subprocess, sys
# Re-establish the environment: `-m embeded.*` resolves the package from the cwd and
# reads EMBEDED_ARTIFACTS from the environment, so a fresh runtime that skipped cell 1
# must not silently train against the in-repo default artifacts dir.
ROOT, ART = '/content/EmbedEd', '/content/embeded-artifacts'
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)   # os.chdir alone does not make `embeded` importable
os.environ.setdefault('EMBEDED_ARTIFACTS', ART)
os.environ.setdefault('EMBEDED_REPORT', ROOT + '/report/measurements.md')
os.environ['PYTHONPATH'] = ROOT + os.pathsep + os.environ.get('PYTHONPATH', '')
import json
from importlib.util import find_spec
if find_spec('embeded.train') is None or find_spec('embeded.evaluate') is None:
    raise RuntimeError('Phase 2 runner modules are missing from this checkout - check REPO_REF in cell 0.')
try:
    from scripts.hf_artifacts import main as hf_main
except Exception as exc:                                    # cell 1 not run in this session
    def hf_main(argv): print('[warn] hf_artifacts unavailable:', exc)

def sh(*args):
    """Run a runner module with its stdout/stderr live. `embeded.train` reports most
    failures as SystemExit(message) + exit 1 with no traceback, so a bare
    CalledProcessError hides the reason; name it here instead."""
    print('$ python -m ' + ' '.join(args), flush=True)
    rc = subprocess.run([sys.executable, '-m', *args]).returncode
    if rc:
        raise SystemExit(
            f'\n`python -m {args[0]}` exited {rc}. The runner printed the reason just above. '
            'The three that exit 1 without a traceback are:\n'
            f'  missing {ART}/fragments.jsonl        -> artifacts are not where EMBEDED_ARTIFACTS points\n'
            '  mining_summary.json was produced by version ...  -> stale artifacts, re-mine (cell 3)\n'
            '  CUDA out of memory                    -> the frozen BATCH/MAX_LEN no longer fit this GPU')

# ---- preflight: fail here, before the first 40-minute run, not inside it -----
from embeded import settings as S
print(f'cwd={os.getcwd()}\nARTIFACTS={S.ARTIFACTS}\nVERSION={S.VERSION}')
for name in ['fragments.jsonl', 'mining_summary.json'] + [f'triples_{c}.jsonl' for c in ('C1', 'C2', 'C3')]:
    path = S.ARTIFACTS / name
    if not path.exists():
        raise SystemExit(f'missing {path} - regenerate (cell 3) or `hf_artifacts pull` before training.')
    if name.endswith('.jsonl'):
        n = sum(1 for _ in open(path, encoding='utf-8'))
        print(f'  {name:<22} {n} lines')
        if n == 0:
            raise SystemExit(f'{path} is empty - mining produced no triples; re-run cell 3.')
    else:
        print(f'  {name:<22} {path.stat().st_size} bytes')
summary = json.loads((S.ARTIFACTS / 'mining_summary.json').read_text())
if (summary.get('_run') or {}).get('version') != S.VERSION:
    raise SystemExit(f"mining_summary.json is version {(summary.get('_run') or {}).get('version')!r} but "
                     f'settings.VERSION is {S.VERSION!r} - stale artifacts; re-mine all three together.')
have = sorted(k for k in summary if not k.startswith('_'))
print('  conditions in mining_summary:', have)
missing = [c for c in ('C1', 'C2', 'C3') if c not in summary]
if missing:
    raise SystemExit(f'{missing} not in mining_summary.json - re-mine all three conditions together.')
import torch
print(f"  cuda={torch.cuda.is_available()} "
      f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(CPU - do not train)'}")
if not torch.cuda.is_available():
    raise SystemExit('No GPU: the frozen 40 min/run budget assumes the T4. Change the runtime type.')

# ---- the nine runs (fail-fast; each resumes from its own checkpoint) ---------
runs = [(c, seed) for c in ('C1', 'C2', 'C3') for seed in (13, 14, 15)]
for condition, seed in runs:
    print(f'\n===== TRAIN {condition} seed={seed} =====', flush=True)
    sh('embeded.train', '--condition', condition, '--seed', str(seed))
    print(f'===== EVALUATE {condition} seed={seed} =====', flush=True)
    sh('embeded.evaluate', '--condition', condition, '--seed', str(seed))
    hf_main(['push', '--if-configured'])  # checkpoint each completed run
print('All nine train/evaluate commands finished.')

## 7. Review results, record deviations, and checkpoint

Before reporting: verify all nine run artifacts and C0; check per-seed F1, precision, recall, MAP@R, and validation-selected thresholds; aggregate mean and spread without selecting the best seed. The test split is for final reporting only. Investigate implausible results as possible harness errors (pair ordering, labels, metrics, checkpoint, threshold code); never tune on test outcomes. Record outcomes and deviations in `report/measurements.md` and append actual completed work to the execution log in `PHASE2_PLAN.md`. Generalisation on BCB s′, UMAP, the demo, and paper/deck work are Phase 3+ and are not done here.

In [ ]:
from scripts.hf_artifacts import main as hf_main
hf_main(['push', '--if-configured', '--include-report'])
print('Review and commit the report and code through the repository workflow; keep large generated weights outside Git.')

## 8. Push the VM's commits back to GitHub

Three separate things make `git push` fail on a Colab VM, and they all surface as **exit 128**: no commit identity (`unable to auto-detect email address` — a fresh VM has none), a dirty worktree (`cannot pull with rebase: You have unstaged changes`), and no credential helper with no terminal to prompt on. The cell below takes them in that order: identity, then commit the tracked `report/` changes (`audit_key*` is never staged), then fetch and rebase onto the branch tip, then a read-only diagnosis, then push.

Identity comes from `GIT_NAME` / `GIT_EMAIL` if you set them, otherwise it is derived from the PAT's own account via `api.github.com`; with neither, the cell stops and says which of the three options to take. Fetching needs no token (the repo is public), so the rebase always runs — without the Secret the cell stops before pushing and you simply re-run it.

Create a **fine-grained PAT** — owner = the account with write access, repository = `ksu-gitreaper807/EmbedEd` only, permission **Contents: Read and write**, short expiry — and put it in Colab's **Secrets** panel (key icon) as `GITHUB_TOKEN`: never in cell source, never in an `os.environ[...] = "ghp_…"` line, never pasted into chat. `scripts.colab_git` stores it in a mode-0600 `~/.git-credentials`, keeps it out of `.git/config` and out of `argv`, and redacts every byte of git output. See `COLAB.md` § "Get results back into the repo".

In [ ]:
import json, os, subprocess, sys, urllib.error, urllib.request

ROOT = '/content/EmbedEd'
REF  = REPO_REF                 # cell 2
GIT_NAME  = ''                  # your GitHub username; '' = derive it from the PAT below
GIT_EMAIL = ''                  # the noreply address shown at https://github.com/settings/emails
COMMIT_DIRTY_REPORT = True      # commit tracked changes under report/ before rebasing
TOKEN_SECRET = 'GITHUB_TOKEN'   # Colab Secrets panel (key icon). No token -> stops before pushing.
API_USER = 'https://api.github.com/user'

def sh(*args, check=False):
    env = {k: v for k, v in os.environ.items()
           if not k.startswith(('GIT_AUTHOR_', 'GIT_COMMITTER_'))}   # these override user.email
    p = subprocess.run(['git', '-C', ROOT, *args], capture_output=True, text=True, env=env)
    shown = (p.stdout + p.stderr).strip()
    print(f"$ git {' '.join(args)} -> exit {p.returncode}")
    if shown:
        print('  ' + shown.replace('\n', '\n  '))
    if check and p.returncode:
        raise SystemExit(f'git {" ".join(args)} failed -- stopping, nothing was lost')
    return p.returncode, p.stdout          # raw: porcelain needs its leading status column

def read_token():                          # Colab Secrets first, then the environment
    try:
        from google.colab import userdata
        value = userdata.get(TOKEN_SECRET)
        if value:
            return value.strip()
    except Exception:
        pass
    return (os.environ.get(TOKEN_SECRET) or '').strip()

# 1. identity first: every commit and every rebase needs one, and a fresh VM has none
if not sh('config', '--get', 'user.email')[1].strip():
    name, email = GIT_NAME, GIT_EMAIL
    token = read_token()
    if not (name and email) and token:
        req = urllib.request.Request(API_USER, headers={'Authorization': f'Bearer {token}',
                                                        'Accept': 'application/vnd.github+json',
                                                        'User-Agent': 'embeded-colab'})
        try:
            with urllib.request.urlopen(req, timeout=20) as resp:
                info = json.loads(resp.read().decode())
            if info.get('id') and info.get('login'):
                name = info['login']
                email = f"{info['id']}+{info['login']}@users.noreply.github.com"
                print(f"  derived the identity from the PAT: {name} <{email}>")
        except (urllib.error.URLError, urllib.error.HTTPError, OSError, ValueError) as exc:
            print(f'  could not reach {API_USER}: {exc.__class__.__name__}')
    if not (name and email):
        raise SystemExit(
            'No git identity on this VM and nothing to derive one from. Pick one:\n'
            f'  a) set GIT_NAME / GIT_EMAIL at the top of this cell, or\n'
            f'  b) add the PAT as a Colab Secret named {TOKEN_SECRET} (then it is derived above), or\n'
            f'  c) run:  !git -C {ROOT} config user.name "you"  and  '
            f'!git -C {ROOT} config user.email "ID+you@users.noreply.github.com"\n'
            'Nothing was changed; re-run this cell afterwards.')
    sh('config', 'user.name', name, check=True)
    sh('config', 'user.email', email, check=True)
    print('  commits will now be authored by: {} <{}>'.format(
        sh('config', '--get', 'user.name')[1].strip(),
        sh('config', '--get', 'user.email')[1].strip()))

# 2. tracked changes block a rebase; untracked strays usually do not
_, raw = sh('status', '--porcelain=v1')
entries = [(line[:2], line[3:].strip()) for line in raw.splitlines() if line.strip()]
tracked = [path for xy, path in entries if xy.strip() != '??']
untracked = [path for xy, path in entries if xy.strip() == '??']

# 3. commit tracked changes under report/ only; audit_key* is never staged
report = [p for p in tracked if p.startswith('report/') and 'audit_key' not in p]
other = [p for p in tracked if p not in report]
if other:
    raise SystemExit(f'tracked changes outside report/ {other} -- commit or stash them, then re-run')
if tracked:
    if not COMMIT_DIRTY_REPORT:
        raise SystemExit(f'tracked changes {tracked} but COMMIT_DIRTY_REPORT is False')
    sh('add', '--', *report, check=True)
    sh('commit', '-m', 'report: VM measurements + audit separability', check=True)

# 4. public repo: fetch needs no token
sh('fetch', 'origin', REF, check=True)

# 5. an untracked file only blocks the rebase if the remote adds that same path
for path in untracked:
    if sh('ls-tree', '--name-only', 'FETCH_HEAD', '--', path)[1].strip():
        raise SystemExit(f'{path} is untracked here but exists on the remote -- move or delete it, '
                         'then re-run')
    print(f'  (untracked, remote does not have it -> leaving it alone: {path})')

# 6. replay any local commit on top of the branch tip
if sh('rebase', 'FETCH_HEAD')[0]:
    raise SystemExit('rebase stopped -- your commits are safe. Either resolve the conflict and '
                     '`git rebase --continue`, or `git -C %s rebase --abort` to undo it.' % ROOT)
sh('log', '--oneline', '-4')

# 7. scripts.colab_git only exists after the rebase above
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
from scripts.colab_git import diagnose, read_secret, sync_and_push

print('\n--- diagnose ---')
for line in diagnose(ROOT, REF):
    print(line)

token = read_secret((TOKEN_SECRET,))
if not token:
    print(f'\nNo {TOKEN_SECRET} in Colab Secrets -> stopping before the push.')
    print('Add the fine-grained PAT as a Colab Secret named GITHUB_TOKEN, then re-run this cell:')
    print('the identity, the commit and the rebase are already done, so it goes straight to push.')
else:
    print('\n--- push ---')
    result = sync_and_push(ROOT, REF, token=token)
    for line in result['lines']:
        print(line)
    print(('ok  ' if result['ok'] else 'FAIL') + f" [{result['stage']}] {result['message']}")
